# MLP on 561 UCI-HAR features  (ICT-4442, owner: Daasharathi Darahaas Yandam)
Follows the shared README protocol: processed data only, seed 42, select on **validation macro-F1**,
test set touched once (final cell), results via `hc.save_results`.

In [ ]:
# Setup
import os, sys, json, time, copy, itertools, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import f1_score, classification_report
import matplotlib; matplotlib.use('Agg') if os.environ.get('SMOKE') else None
import matplotlib.pyplot as plt

if not os.environ.get('SMOKE'):
    from google.colab import drive; drive.mount('/content/drive')
    sys.path.insert(0, '/content/drive/MyDrive/DL PROJ')   # adjust if your folder is elsewhere
else:
    sys.path.insert(0, os.environ['SMOKE_DIR'])
import har_common as hc

SEED = 42
MAX_EPOCHS = int(os.environ.get('MAX_EPOCHS', 200))   # protocol: 200
PATIENCE   = int(os.environ.get('PATIENCE', 20))      # protocol: >= 20
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

ROOT = hc.find_project_root(); OUT = ROOT / 'mlp'
for sub in ('final_model', 'figures'): (OUT / sub).mkdir(parents=True, exist_ok=True)
print('device:', dev, '| out:', OUT)

In [ ]:
# Data (never refit / re-split)
d = hc.load_data('tabular')
print({k: d[k].shape for k in ('X_train', 'X_val', 'X_test')}, '| dataset_version:', d['meta']['dataset_version'])
T = lambda a, dt: torch.as_tensor(a, dtype=dt, device=dev)
Xtr, ytr = T(d['X_train'], torch.float32), T(d['y_train'], torch.long)
Xva, yva = T(d['X_val'],   torch.float32), T(d['y_val'],   torch.long)
Xte, yte = T(d['X_test'],  torch.float32), T(d['y_test'],  torch.long)

In [ ]:
# Model + training
def make_model(cfg):
    layers, i = [], 561
    for h in cfg['hidden']:
        layers += [nn.Linear(i, h), nn.ReLU(), nn.Dropout(cfg['dropout'])]; i = h
    return nn.Sequential(*layers, nn.Linear(i, 6))

@torch.no_grad()
def evaluate(model, X, y):
    """Eval mode, plain cross-entropy (comparable across models)."""
    model.eval(); logits = model(X)
    loss = F.cross_entropy(logits, y).item(); pred = logits.argmax(1)
    p = F.softmax(logits, 1).cpu().numpy()
    return dict(loss=loss, acc=(pred == y).float().mean().item(),
                f1=f1_score(y.cpu().numpy(), pred.cpu().numpy(), average='macro'), proba=p)

def train(cfg, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    set_seed(); model = make_model(cfg).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    sched = ReduceLROnPlateau(opt, 'max', factor=0.5, patience=8) if cfg.get('sched') == 'plateau' else None
    g = torch.Generator().manual_seed(SEED); n, bs = len(Xtr), cfg['bs']
    if dev == 'cuda': torch.cuda.reset_peak_memory_stats()
    best, best_state, best_ep, bad, hist = -1, None, 0, 0, []
    for ep in range(1, max_epochs + 1):
        t0 = time.time(); model.train()
        perm = torch.randperm(n, generator=g).to(dev)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        tr, va = evaluate(model, Xtr, ytr), evaluate(model, Xva, yva)
        if sched: sched.step(va['f1'])
        if dev == 'cuda': torch.cuda.synchronize()
        hist.append(dict(epoch=ep, train_loss=tr['loss'], train_acc=tr['acc'], train_macro_f1=tr['f1'],
                         val_loss=va['loss'], val_acc=va['acc'], val_macro_f1=va['f1'],
                         lr=opt.param_groups[0]['lr'], epoch_time_s=time.time() - t0))
        if va['f1'] > best:
            best, best_ep, bad = va['f1'], ep, 0; best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)                      # restore best-epoch weights
    h = pd.DataFrame(hist)
    return dict(model=model, hist=h, best_f1=best, best_ep=best_ep, val_acc=h.val_acc[best_ep - 1],
                epochs=len(h), time=h.epoch_time_s.sum(),
                peak_mb=torch.cuda.max_memory_allocated() / 2**20 if dev == 'cuda' else np.nan)

In [ ]:
# Baseline (untuned): 561-128-64-6, dropout .3, AdamW 1e-3, wd .01, bs 64
BASE = dict(hidden=(128, 64), dropout=0.3, lr=1e-3, wd=0.01, bs=64, sched='none')
base = train(BASE)
print(f"baseline: val macro-F1 {base['best_f1']:.4f} @ epoch {base['best_ep']} ({base['epochs']} epochs)")

In [ ]:
# Grid search - validation macro-F1 only (Stage A: architecture, Stage B: recipe)
grid_rows = []
def run_cfg(cfg, stage):
    r = train(cfg)
    grid_rows.append(dict(stage=stage, hidden=str(cfg['hidden']), dropout=cfg['dropout'], lr=cfg['lr'],
                          wd=cfg['wd'], bs=cfg['bs'], sched=cfg['sched'], val_macro_f1=r['best_f1'],
                          val_acc=r['val_acc'], best_epoch=r['best_ep'], epochs_trained=r['epochs'],
                          train_time_s=r['time']))
    print(f"[{stage}] {cfg['hidden']} do={cfg['dropout']} lr={cfg['lr']} wd={cfg['wd']} {cfg['sched']}"
          f" -> val F1 {r['best_f1']:.4f}")
    return r['best_f1']

# Stage A
archs = [(128, 64), (256, 128), (512, 256), (512, 256, 128)]
scoresA = {}
for h, do in itertools.product(archs, [0.2, 0.3, 0.5]):
    cfg = {**BASE, 'hidden': h, 'dropout': do}; scoresA[(h, do)] = run_cfg(cfg, 'A_architecture')
(bestA_h, bestA_do) = max(scoresA, key=scoresA.get)
# Stage B (on best architecture)
scoresB = {}
for lr, wd, sc in itertools.product([1e-3, 3e-4], [0.01, 0.1], ['none', 'plateau']):
    cfg = {**BASE, 'hidden': bestA_h, 'dropout': bestA_do, 'lr': lr, 'wd': wd, 'sched': sc}
    scoresB[(lr, wd, sc)] = run_cfg(cfg, 'B_recipe')
bestB = max(scoresB, key=scoresB.get)
TUNED = {**BASE, 'hidden': bestA_h, 'dropout': bestA_do, 'lr': bestB[0], 'wd': bestB[1], 'sched': bestB[2]}
if max(scoresB.values()) < scoresA[(bestA_h, bestA_do)]:   # stage B never beat stage A's setting
    TUNED = {**BASE, 'hidden': bestA_h, 'dropout': bestA_do}
grid = pd.DataFrame(grid_rows); grid.to_csv(OUT / 'grid_search_results.csv', index=False)
print('TUNED config:', TUNED)

for stage, fn, ttl in (('A_architecture', 'grid_search_stageA_architecture.png', 'Stage A: architecture x dropout'),
                       ('B_recipe', 'grid_search_stageB_recipe.png', 'Stage B: lr x wd x schedule')):
    g_ = grid[grid.stage == stage].reset_index(drop=True)
    lab = [f"{r.hidden}\ndo={r.dropout}" if stage[0] == 'A' else f"lr={r.lr}\nwd={r.wd}\n{r.sched}" for r in g_.itertuples()]
    fig, ax = plt.subplots(figsize=(max(8, .8 * len(g_)), 4.5))
    ax.bar(range(len(g_)), g_.val_macro_f1, color='tab:blue'); ax.set_xticks(range(len(g_)), lab, fontsize=7)
    ax.set_ylim(max(0, g_.val_macro_f1.min() - .02), g_.val_macro_f1.max() + .01)
    ax.set_ylabel('validation macro-F1'); ax.set_title(ttl); ax.grid(alpha=.3, axis='y')
    fig.tight_layout(); fig.savefig(OUT / 'figures' / fn, dpi=150); plt.show()

In [ ]:
# Final tuned run (fresh run with the chosen config)
tuned = train(TUNED)
print(f"tuned: val macro-F1 {tuned['best_f1']:.4f} @ epoch {tuned['best_ep']} ({tuned['epochs']} epochs)")

In [ ]:
# FINAL EVALUATION - first and only time the test set is used
OWNER = 'Daasharathi Darahaas Yandam'
def finalize(tag, res, cfg):
    model = res['model'].eval()
    P = {k: evaluate(model, X, y)['proba'] for k, (X, y) in dict(tr=(Xtr, ytr), va=(Xva, yva), te=(Xte, yte)).items()}
    w = OUT / 'final_model' / f'mlp_{tag}_weights.pt'; torch.save(model.state_dict(), w)
    eff = dict(params_total=hc.count_params(model), model_size_mb=hc.file_size_mb(w),
               flops_per_window=hc.count_flops(model, (1, 561)),
               latency_ms_cpu_b1=hc.measure_latency(model, (1, 561), 'cpu'),
               latency_ms_gpu_b1=hc.measure_latency(model, (1, 561), 'cuda') if dev == 'cuda' else np.nan,
               throughput_gpu_wps=hc.measure_throughput(model, (561,), 256, 'cuda') if dev == 'cuda' else np.nan,
               peak_gpu_mem_mb=res['peak_mb'])
    notes = (f"MLP 561-{'-'.join(map(str, cfg['hidden']))}-6, ReLU, dropout {cfg['dropout']}; AdamW lr={cfg['lr']} "
             f"wd={cfg['wd']} bs={cfg['bs']} sched={cfg['sched']}; plain CE; no augmentation/class weights; "
             f"FLOPs: torch.utils.flop_counter")
    row = hc.save_results(OUT / 'mlp_results.csv', model_name='mlp', family='MLP', owner=OWNER,
        input_representation='tabular_561', run_tag=tag,
        y_train=d['y_train'], proba_train=P['tr'], y_val=d['y_val'], proba_val=P['va'],
        y_test=d['y_test'], proba_test=P['te'], subj_test=d['subj_test'], efficiency=eff,
        training=dict(train_time_s=res['time'], epochs_trained=res['epochs'], best_epoch=res['best_ep'],
                      time_per_epoch_s=res['time'] / res['epochs']), notes=notes)
    res['hist'].to_csv(OUT / ('training_history_final.csv' if tag == 'tuned' else 'training_history_baseline.csv'), index=False)
    return row, P

row_b, P_b = finalize('baseline', base, BASE)
row_t, P_t = finalize('tuned', tuned, TUNED)
assert row_t['dataset_version'] == d['meta']['dataset_version']
print(pd.DataFrame([row_b, row_t])[['run_tag', 'test_accuracy', 'test_macro_f1', 'val_macro_f1', 'train_macro_f1',
                                    'params_total', 'flops_per_window', 'latency_ms_cpu_b1']].to_string())

In [ ]:
# Remaining files (tuned model)
P_tr, P_va, P_te = P_t['tr'], P_t['va'], P_t['te']; y_te = d['y_test']; pred = P_te.argmax(1)
(OUT / 'classification_report_test.txt').write_text(
    classification_report(y_te, pred, target_names=hc.CLASS_NAMES, digits=4))
ps = (pd.DataFrame(dict(subject=d['subj_test'], correct=(pred == y_te))).groupby('subject').correct.mean()
      .rename('accuracy').reset_index())
ps.to_csv(OUT / 'per_subject_accuracy_test.csv', index=False)
np.savez(OUT / 'final_model' / 'predictions.npz', P_train=P_tr, P_val=P_va, P_test=P_te,
         y_test=y_te, subj_test=d['subj_test'])
json.dump({**{k: (list(v) if isinstance(v, tuple) else v) for k, v in TUNED.items()},
           'seed': SEED, 'max_epochs': MAX_EPOCHS, 'patience': PATIENCE, 'optimizer': 'AdamW',
           'loss': 'cross_entropy', 'input': 'tabular_561', 'dataset_version': d['meta']['dataset_version']},
          open(OUT / 'final_model' / 'config.json', 'w'), indent=2)

keys = ['test_accuracy', 'test_macro_f1', 'test_macro_precision', 'test_macro_recall', 'test_weighted_f1',
        'test_balanced_accuracy', 'test_cohen_kappa', 'test_mcc', 'test_log_loss', 'test_roc_auc_macro_ovr',
        'test_top2_accuracy', 'sit_stand_confusion_rate', 'walking_variants_confusion_rate',
        'static_dynamic_confusion_rate', 'subject_acc_mean', 'subject_acc_min']
bt = pd.DataFrame(dict(metric=keys, baseline=[row_b[k] for k in keys], tuned=[row_t[k] for k in keys]))
bt['change'] = bt.tuned - bt.baseline; bt.to_csv(OUT / 'baseline_vs_tuned_test.csv', index=False); print(bt.to_string())

In [ ]:
# Figures
FIG = OUT / 'figures'
hc.plot_confusion_matrix(y_te, pred, FIG / 'confusion_matrix_test.png', 'Test')
hc.plot_confusion_matrix(d['y_val'], P_va.argmax(1), FIG / 'confusion_matrix_val.png', 'Validation')

h = pd.read_csv(OUT / 'training_history_final.csv'); best = int(h.val_macro_f1.idxmax()) + 1
fig, ax = plt.subplots(1, 4, figsize=(22, 4.6))
ax[0].plot(h.epoch, h.train_loss, label='train'); ax[0].plot(h.epoch, h.val_loss, label='validation')
ax[0].set_yscale('log'); ax[0].set_title('Cross-entropy loss')
ax[1].plot(h.epoch, h.train_acc, label='train'); ax[1].plot(h.epoch, h.val_acc, label='validation'); ax[1].set_title('Accuracy')
ax[2].plot(h.epoch, h.train_macro_f1, label='train'); ax[2].plot(h.epoch, h.val_macro_f1, label='validation'); ax[2].set_title('Macro-F1')
ax[3].plot(h.epoch, h.lr); ax[3].set_yscale('log'); ax[3].set_title('Learning rate')
for a in ax: a.axvline(best, ls='--', c='gray'); a.set_xlabel('epoch'); a.grid(alpha=.3)
for a in ax[:3]: a.legend()
fig.tight_layout(); fig.savefig(FIG / 'learning_curves_final.png', dpi=150); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(ps.subject.astype(str), ps.accuracy); ax[0].axhline(ps.accuracy.mean(), ls='--', c='k')
ax[0].set_title('Test accuracy per subject'); ax[0].set_xlabel('subject'); ax[0].set_ylim(max(0, ps.accuracy.min() - .05), 1.0)
f1s = [row_t[f'f1_{c}'] for c in hc.CLASS_NAMES]
ax[1].bar(['WALK', 'UP', 'DOWN', 'SIT', 'STAND', 'LAY'], f1s); ax[1].set_title('Test F1 per class'); ax[1].set_ylim(max(0, min(f1s) - .05), 1.0)
for a in ax: a.grid(alpha=.3, axis='y')
fig.tight_layout(); fig.savefig(FIG / 'per_subject_and_per_class.png', dpi=150); plt.show()

hb = pd.read_csv(OUT / 'training_history_baseline.csv')
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for hh, lab in ((hb, 'baseline'), (h, 'tuned')):
    ax[0].plot(hh.epoch, hh.val_loss, label=lab); ax[1].plot(hh.epoch, hh.val_macro_f1, label=lab)
ax[0].set_yscale('log'); ax[0].set_title('Validation loss'); ax[1].set_title('Validation macro-F1')
for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3); a.legend()
fig.tight_layout(); fig.savefig(FIG / 'baseline_vs_tuned_curves.png', dpi=150); plt.show()
print(sorted(p.name for p in OUT.rglob('*') if p.is_file()))